# Image classification with quantum kernels in QARP

A quantum kernel measures the similarity of two data points through the quantum states
they are encoded into. Paired with a standard SVM, it is a drop-in kernel for
classification.

This notebook classifies Fashion-MNIST images with a **projected quantum kernel**
(Huang et al., [Nat. Commun. 12, 2631 (2021)](https://www.nature.com/articles/s41467-021-22539-9)),
built end to end in QARP:

1. encode each image with a QARP feature-map circuit,
2. estimate the kernel by measurement — classical shadows via `PauliShadow`,
3. train an SVM on the resulting kernel and classify held-out images.

In [ ]:
%pip install -q scikit-learn scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import qarpx as qx                                    # for GateType (RZZ, Rz, ...)
from qarp.blocks import LayerBlock, CompositeBlock    # QARP layer blocks
from qarp.algorithms import PauliShadow               # classical-shadow measurement primitive
from qarp.engines import QarpEngine
from qarp.operators import QubitOperator
from sklearn.metrics.pairwise import rbf_kernel

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

## Settings

Pick one run size. `DEMO` runs in about a minute. `FULL_SCALE` is the paper's experiment
size (N = 600). `N_TRAIN = None` uses every image of the two classes (~12k) — the full
Fashion-MNIST binary task, but the kernel and SVM are O(N²) plus one measurement campaign
per image, so expect a long run and a few GB of memory.

In [ ]:
N_QUBITS, ENCODING = 8, "E3"

# DEMO -- fast, for a quick run (~1 min)
N_TRAIN, N_TEST, N_SHADOW = 60, 30, 600

# FULL_SCALE -- the paper's experiment size, N = 600 (~5-10 min)
# N_TRAIN, N_TEST, N_SHADOW = 600, 200, 2000

# ENTIRE two-class Fashion-MNIST -- N_TRAIN=None takes all ~12k images (hours, O(N^2) memory)
# N_TRAIN, N_TEST, N_SHADOW = None, None, 2000

print(f"{'ALL' if N_TRAIN is None else N_TRAIN} train / "
      f"{'auto' if N_TEST is None else N_TEST} test images, {N_SHADOW} measurements per image.")

## Dataset

Fashion-MNIST restricted to two classes, T-shirt and Trouser, as 28×28 grayscale images.
We hold out a test set and fit PCA and scaling on the training set only.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

CLASS_A, CLASS_B = 0, 1                     # 0 = T-shirt/top, 1 = Trouser
NAMES = {0: "T-shirt", 1: "Trouser"}
fm = fetch_openml("Fashion-MNIST", version=1, as_frame=False)
keep = np.isin(fm.target.astype(int), [CLASS_A, CLASS_B])
imgs = fm.data[keep]
y = (fm.target.astype(int)[keep] == CLASS_B).astype(int)

n_total = len(imgs)
n_test = N_TEST if N_TEST is not None else n_total // 5      # None -> 20% held out
n_train = N_TRAIN if N_TRAIN is not None else n_total - n_test  # None -> all the rest
ii = np.random.default_rng(0).permutation(n_total)[:n_train + n_test]
imgs, y = imgs[ii], y[ii]
raw_tr, raw_te = imgs[:n_train], imgs[n_train:]
y_tr, y_te = y[:n_train], y[n_train:]

pca = PCA(N_QUBITS, random_state=0).fit(raw_tr)
scaler = MinMaxScaler().fit(pca.transform(raw_tr))
X_tr = scaler.transform(pca.transform(raw_tr))
X_te = scaler.transform(pca.transform(raw_te))

fig, axes = plt.subplots(2, 6, figsize=(7.2, 2.7))
for row, cls in enumerate((CLASS_A, CLASS_B)):
    for ax, img in zip(axes[row], raw_tr[y_tr == (cls == CLASS_B)][:6], strict=False):
        ax.imshow(img.reshape(28, 28), cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_visible(False)
    axes[row, 0].set_ylabel(NAMES[cls], rotation=0, ha="right", va="center", fontsize=11)
plt.tight_layout(); plt.show()
print(f"{n_train} train / {n_test} test images, reduced to {N_QUBITS} features.")

## Feature map

Each image is encoded into a state $|\phi(x)\rangle = U(x)\,|0\rangle^{\otimes n}$ by a ZZ
feature map, built from QARP layer blocks: a Hadamard layer, a data-dependent $R_z$ layer,
and $R_{zz}$ entangling layers. `ENCODING` (E1–E3) sets the entangling depth.

In [ ]:
def feature_layers(x, n=N_QUBITS, encoding=ENCODING):
    '''U(x) as a list of QARP LayerBlocks (a ZZ feature map). RZZ(t) == cx.rz(t).cx.'''
    x = np.asarray(x, float); pi = float(np.pi)
    L = [LayerBlock(qx.GateType.H, n),
         LayerBlock(qx.GateType.Rz, n, parameters=[pi * x[k % len(x)] for k in range(n)])]
    if encoding in ("E2", "E3"):
        L.append(LayerBlock(qx.GateType.RZZ, n, overlapping=1,
                 parameters=[pi * x[k % len(x)] * x[(k + 1) % len(x)] for k in range(n - 1)]))
    if encoding == "E3":
        L.append(LayerBlock(qx.GateType.Rz, n, parameters=[pi * x[k % len(x)] for k in range(n)]))
        L.append(LayerBlock(qx.GateType.RZZ, n, overlapping=0,
                 parameters=[pi * x[(k + 1) % len(x)] ** 2 for k in range(0, n - 1, 2)]))
    return L

def feature_block(x, n=N_QUBITS, encoding=ENCODING):
    return CompositeBlock(feature_layers(x, n, encoding), n_qubits=n, name="featuremap")

## Measuring each image with classical shadows

The projected kernel describes each encoded state by its single-qubit **reduced density
matrices** $\rho_k(x) = \mathrm{tr}_{j\neq k}\,|\phi(x)\rangle\langle\phi(x)|$.
`PauliShadow` runs one random-measurement campaign per image and reconstructs the local
expectations $\langle X_k\rangle,\ \langle Y_k\rangle,\ \langle Z_k\rangle$ for every
qubit from it — the same routine one would run on hardware.

In [ ]:
def shadow_features(X, encoding=ENCODING, n_shadow=N_SHADOW, seed0=0):
    '''Per-image local expectations [<X_0>,<Y_0>,<Z_0>, <X_1>,...] from PauliShadow.'''
    prims = [PauliShadow(QubitOperator("Z0"), feature_block(x, encoding=encoding),
                         n_settings=n_shadow, seed=seed0 + i) for i, x in enumerate(X)]
    eng = QarpEngine(seed=seed0); eng.build(prims); eng.run()
    return np.array([[p.dataset.estimator().expval(f"{P}{k}").value
                      for k in range(N_QUBITS) for P in ("X", "Y", "Z")] for p in prims])

feats_tr = shadow_features(X_tr, seed0=1)
feats_te = shadow_features(X_te, seed0=10_000)
print(f"measured {len(feats_tr)} + {len(feats_te)} images; each -> {feats_tr.shape[1]} local expectations.")

### From shadows to the density matrices $\rho_k$

A single qubit's reduced density matrix is fixed by its three expectations:

$$\rho_k = \tfrac{1}{2}\left(I + \langle X_k\rangle\,X + \langle Y_k\rangle\,Y + \langle Z_k\rangle\,Z\right).$$

So the measured features *are* the $\rho_k$, written in the Pauli basis. We reconstruct
them for one image and check they are valid density matrices (Hermitian, unit trace,
non-negative eigenvalues; shadow estimates are approximate, so tiny violations are
expected at finite measurements).

In [ ]:
I2 = np.eye(2); X = np.array([[0,1],[1,0]], complex)
Y = np.array([[0,-1j],[1j,0]], complex); Z = np.array([[1,0],[0,-1]], complex)

def rho_k(feature_vec):
    '''List of single-qubit density matrices rho_k from a feature vector.'''
    r = np.asarray(feature_vec).reshape(N_QUBITS, 3)          # rows: (<X_k>, <Y_k>, <Z_k>)
    return [0.5 * (I2 + xk*X + yk*Y + zk*Z) for xk, yk, zk in r]

rho = rho_k(feats_tr[0])
r0 = rho[0]
print("rho_0 for the first training image:")
print(np.round(r0, 3))
print(f"trace = {np.trace(r0).real:.3f}   hermitian = {np.allclose(r0, r0.conj().T)}"
      f"   eigenvalues = {np.round(np.linalg.eigvalsh(r0).real, 3)}")

plt.figure(figsize=(3.6, 2.8))
plt.imshow(feats_tr[0].reshape(N_QUBITS, 3), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(fraction=0.046); plt.xticks(range(3), ["⟨X⟩", "⟨Y⟩", "⟨Z⟩"])
plt.yticks(range(N_QUBITS), [f"q{k}" for k in range(N_QUBITS)])
plt.title("Reduced density matrices of one image", fontsize=10); plt.tight_layout(); plt.show()

### The kernel

The projected kernel is a Gaussian in the total distance between two images' density
matrices:

$$k(x, x') = \exp\!\left(-\gamma \sum_k \left\lVert \rho_k(x) - \rho_k(x') \right\rVert_F^2\right).$$

In the Pauli basis that distance is just the squared distance between the feature
vectors, so the kernel is an RBF kernel on `feats`.

In [ ]:
gamma = 1.0 / feats_tr.shape[1]
K_tr = rbf_kernel(feats_tr, feats_tr, gamma=gamma)     # train x train
K_te = rbf_kernel(feats_te, feats_tr, gamma=gamma)     # test  x train

order = np.argsort(y_tr)
plt.figure(figsize=(4.2, 3.6))
plt.imshow(K_tr[order][:, order], cmap="viridis")
plt.title("Quantum kernel matrix (train, ordered by class)", fontsize=10)
plt.colorbar(fraction=0.046); plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()

## Classify

The kernel is precomputed, so it plugs straight into scikit-learn's `SVC(kernel="precomputed")`.
We fit on the training kernel and predict the held-out images.

In [ ]:
from sklearn.svm import SVC

clf = SVC(kernel="precomputed").fit(K_tr, y_tr)
pred = clf.predict(K_te)
acc = float((pred == y_te).mean())
print(f"quantum-kernel SVM test accuracy: {acc:.2f}")

fig, axes = plt.subplots(2, 6, figsize=(7.2, 3.6))
for ax, img, p, t in zip(axes.ravel(), raw_te, pred, y_te, strict=False):
    ax.imshow(img.reshape(28, 28), cmap="gray_r"); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_title(NAMES[CLASS_B if p else CLASS_A], fontsize=8,
                 color=("#2e8b57" if p == t else "#c0392b"))
fig.suptitle("Predictions on held-out images (red = wrong)", fontsize=10)
plt.tight_layout(); plt.subplots_adjust(top=0.86, hspace=0.55); plt.show()

## Summary

We classified Fashion-MNIST images with a projected quantum kernel, built entirely from
QARP components: a `LayerBlock`/`CompositeBlock` feature map, `PauliShadow` to measure the
kernel, and a scikit-learn SVM on the precomputed kernel. The same code path runs on
hardware, since the kernel comes from measurements rather than the statevector.

To scale up, switch the settings cell from `DEMO` to `FULL_SCALE`. Other feature maps
(`ENCODING`), class pairs (`CLASS_A`, `CLASS_B`), or kernels are drop-in changes.

*See Huang et al. for the theory of quantum kernels and when they may help — a separate
question from the classification workflow shown here.*